# CommuteCast — Stage 5: Integrated Commute Analysis Layer

## Unit-1 Mathematical Integration & Analytical Separation

In this stage, we bring together the two foundational Unit-1 components developed in Stages 3 and 4 into a single, cohesive analysis framework:

1. **Polynomial Regression Component**:  
   $$\text{Expected Traffic Volume} = f(\text{Hour})$$  
   Estimates the typical traffic volume (in vehicles/hour) for any selected hour of the day using our selected Degree-3 polynomial curve.

2. **Empirical Probability Component**:  
   $$P(\text{High} \mid \text{Conditions})$$  
   Estimates the empirical likelihood that traffic volume falls into the upper quartile ($Q_{75} = 4,933.0$ veh/hr) under specific environmental and temporal conditions (Rain / Dry, Peak / Off-Peak).

### Core Design Principles
- **Mathematical Separation**: These two components answer fundamentally distinct questions. We strictly avoid multiplying or blending them into an arbitrary composite index.
- **No Opaque ML Algorithms**: Fully explainable, grounded in Unit-1 calculus, polynomial regression, and classical probability.
- **Rule-Based Interpretation**: We compare conditional probabilities directly against the empirical baseline $P(\text{High}) = 24.98\%$ without inventing ad-hoc probability thresholds.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.integration import (
    CommuteAnalysisEngine,
    analyze_commute_conditions,
    format_readable_analysis,
    get_commute_engine
)

print("Stage 5 Integration modules loaded successfully.")

## 1. Engine Initialization & Dataset Baseline

The analysis engine calculates the global dataset baseline $P(\text{High})$ and the four empirical condition quadrants from the cleaned historical dataset ($N = 48,187$ observations).

In [ ]:
engine = get_commute_engine()

print(f"Total Cleaned Observations: {engine.count_total:,}")
print(f"High Congestion Threshold (Q75): 4,933.0 veh/hr")
print(f"High Congestion Observations: {engine.count_high:,}")
print(f"Baseline P(High): {engine.p_high_baseline:.4f} ({engine.p_high_baseline * 100:.2f}%)")

## 2. Four Empirical (Rain, Peak) Quadrants

Rather than fabricating probabilities, the engine looks up exact empirical frequencies across the 4 joint combinations of weather and commute timing:

In [ ]:
quad_data = []
for (rain_val, peak_val), data in engine.joint_probabilities.items():
    quad_data.append({
        "Rain": "Rain" if rain_val == 1 else "No Rain",
        "Commute Peak": "Peak" if peak_val == 1 else "Off-Peak",
        "Sample Count (N)": f"{data['n_obs']:,}",
        "High Congestion Count": f"{data['n_high']:,}",
        "P(High | Condition)": f"{data['prob']:.4f} ({data['prob']*100:.2f}%)",
        "Difference from Baseline": f"{(data['prob'] - engine.p_high_baseline)*100:+.2f} ppt",
        "Subset Mean Traffic": f"{data['expected_traffic_subset']:.1f} veh/hr"
    })

quad_df = pd.DataFrame(quad_data)
display(quad_df)

## 3. Bayes' Theorem Educational Exposure

The engine can unpack how Bayes' Theorem reconstructs conditional probability from marginal probabilities and likelihoods:
$$P(\text{High} \mid \text{Rain}) = \frac{P(\text{Rain} \mid \text{High}) \cdot P(\text{High})}{P(\text{Rain})}$$

In [ ]:
bayes_demo = engine.explain_bayes("rain")
print("=== Educational Bayes' Theorem Decomposition ===")
print(f"Evidence (B): {bayes_demo['evidence_name']}")
print(f"Prior P(High):            {bayes_demo['prior_P_High']:.6f} ({bayes_demo['prior_P_High']*100:.2f}%)")
print(f"Likelihood P(Rain | High): {bayes_demo['likelihood_P_Evidence_given_High']:.6f} ({bayes_demo['likelihood_P_Evidence_given_High']*100:.2f}%)")
print(f"Marginal P(Rain):         {bayes_demo['marginal_P_Evidence']:.6f} ({bayes_demo['marginal_P_Evidence']*100:.2f}%)")
print(f"Reconstructed P(High|Rain):{bayes_demo['bayes_P_High_given_Evidence']:.6f} ({bayes_demo['bayes_P_High_given_Evidence']*100:.2f}%)")
print(f"Direct Empirical P(High|Rain): {bayes_demo['direct_empirical_P_High_given_Evidence']:.6f} ({bayes_demo['direct_empirical_P_High_given_Evidence']*100:.2f}%)")
print(f"Discrepancy:              {bayes_demo['exact_match_discrepancy']:.2e} (Exact Empirical Match)")

## 4. Reusable Analysis Function: Example Execution

Here we run the functional entry point `analyze_commute_conditions` for 8:00 AM under rainy conditions and format the human-readable report.

In [ ]:
sample_analysis = analyze_commute_conditions(hour=8, rain=1, peak=1)
readable_report = format_readable_analysis(sample_analysis)
print(readable_report)

## 5. Systematic Evaluation of the 8 Required Scenarios

We evaluate the integrated engine across 8 core operational scenarios spanning morning and evening rush hours with and without precipitation:
1. 7 AM + No Rain
2. 7 AM + Rain
3. 8 AM + No Rain
4. 8 AM + Rain
5. 4 PM (16:00) + No Rain
6. 4 PM (16:00) + Rain
7. 5 PM (17:00) + No Rain
8. 5 PM (17:00) + Rain

In [ ]:
scenarios = [
    (7, 0, "7 AM + No Rain"),
    (7, 1, "7 AM + Rain"),
    (8, 0, "8 AM + No Rain"),
    (8, 1, "8 AM + Rain"),
    (16, 0, "4 PM (16:00) + No Rain"),
    (16, 1, "4 PM (16:00) + Rain"),
    (17, 0, "5 PM (17:00) + No Rain"),
    (17, 1, "5 PM (17:00) + Rain")
]

rows = []
for hour_val, rain_val, desc in scenarios:
    res = analyze_commute_conditions(hour=hour_val, rain=rain_val)
    rows.append({
        "Scenario": desc,
        "Hour": f"{res['hour']:02d}:00",
        "Rain": "Rain (1)" if res["rain"] == 1 else "No Rain (0)",
        "Peak": "Peak (1)" if res["peak"] == 1 else "Off-Peak (0)",
        "Expected Volume (veh/hr)": f"{res['expected_traffic']:,.1f}",
        "P(High | Conditions)": f"{res['high_congestion_probability']*100:.2f}%",
        "Baseline P(High)": f"{res['baseline_high_probability']*100:.2f}%",
        "Diff from Baseline": f"{res['difference_percentage_points']:+.2f} ppt",
        "Interpretation": res["interpretation"]
    })

scenario_df = pd.DataFrame(rows)
display(scenario_df)

## 6. How the Two Components Complement Each Other

The polynomial model and the probability model provide two distinct lenses on the traffic system:

| Dimension | Polynomial Regression Component | Empirical Probability Component |
| :--- | :--- | :--- |
| **Mathematical Expression** | $\hat{y} = f(\text{Hour})$ (Degree 3 Cubic) | $P(\text{High} \mid \text{Rain}, \text{Peak})$ |
| **Core Question Answered** | *"How many vehicles typically traverse I-94 during this hour?"* | *"What is the historical risk that volume exceeds the severe threshold ($Q_{75}$)?"* |
| **Output Nature** | Continuous expected volume (veh/hr) | Discrete conditional probability ($[0, 1]$) |
| **Sensitivity** | Captures smooth diurnal flow cycles (diurnal peak at 4 PM) | Captures compounding extreme event likelihood (e.g. rain during peak) |
| **Synergy** | Establishes the expected baseline demand | Quantifies the volatility and tail-risk around that demand |

By keeping these components separate, CommuteCast delivers clear, transparent insights without obscuring them behind a black-box machine learning classifier.

## 7. Statistical Limitations & Governance

1. **Single Feature for Polynomial Regression**: The polynomial curve models traffic volume solely as a function of the hour of the day. It does not account for day of the week, seasonality, or sudden road closures.
2. **Empirical Historical Basis**: Probabilities are strictly derived from historical observations. They represent empirical frequencies rather than future guarantees.
3. **Association is Not Causation**: A higher probability of high congestion during rainy conditions ($67.75\%$ vs. $65.31\%$) reflects an empirical statistical association. It does not prove that rain causally produces congestion.
4. **In-Sample Evaluation**: The polynomial metrics were computed on 24 aggregated hourly means. While representative of long-term diurnal trends, individual hourly deviations may have higher variance.
5. **Interpretability over Black-Box ML**: The system is designed as an explainable, transparent analytical tool adhering strictly to Unit-1 principles, rather than an uninterpretable deep neural network or complex ensemble.